In [ ]:
ZZ

In [ ]:
QQ

In [ ]:
RR

In [ ]:
CC

In [ ]:
RDF

In [ ]:
a=CC(sqrt(2))

In [ ]:
type(a)

In [ ]:
ComplexField(200)(sqrt(2))

In [ ]:
var('x y')
f = sin(x)**4 + cos(x)**4
f.simplify_full()

In [ ]:
diff(exp(-x^2), x)

In [ ]:
integrate(exp(-x^2), x)

In [ ]:
taylor(exp(-x^2), x, 0, 6)

In [ ]:
solve(x^2 + 2*x + 1 == 0, x)

In [ ]:
solve(x^2 + y*x + 1 == 0, x)

In [ ]:
R.<x> = PolynomialRing(QQ)

In [ ]:
p = (x^5-1)/(x-1)
p.factor()

In [ ]:
F = GF(7)
F(3)^(-1)

In [ ]:
K.<a> = GF(7^2, modulus=x^2+1)
a^2

In [ ]:
parent(x)

In [ ]:
plot(sin(x)/x, (x,0,10))

In [ ]:
# 1) Choose exact domains first
# QQ, ZZ = RationalField(), IntegerRing()

# 2) Create the right ring/field explicitly
R.<x> = PolynomialRing(QQ)

# 3) Do algebraic manipulations exactly
p = x^6 - 1
fac = p.factor()
print(fac)

# 4) Only at the end: numerical evaluation
RR200 = RealField(200)
val = RR200(fac[0][0](RR200(pi)))
val


In [ ]:
ZZ()

In [1]:
from ore_algebra import *
Pol.<u> = QQ[]
Dop.<Du> = OreAlgebra(Pol)

In [ ]:
from sage.all import *
from sage.libs.gap.libgap import libgap
from sage.modular.arithgroup.arithgroup_perm import (
    sl2z_word_problem,              # expresses a matrix as a word in (l,r)
    ArithmeticSubgroup_Permutation,  # builds subgroup from coset permutations
)

def subgroup_from_matrix_gens_farey(gens, relabel=True, sanity_check=True):
    """
    Build the *projective* subgroup <gens> in PSL2(Z) from SL2(Z) matrix generators,
    as an ArithmeticSubgroup_Permutation, then return (H, F) where F = H.farey_symbol().

    WARNING: requires the subgroup to have finite index in PSL2(Z), otherwise coset enumeration may not terminate.
    """
    # --- 0) Basic validation of generators ---
    gensM = []
    for A in gens:
        A = matrix(ZZ, A)
        if A.nrows() != 2 or A.ncols() != 2 or A.det() != 1:
            raise ValueError(f"Generator is not in SL(2, Z):\n{A}")
        gensM.append(A)

    # --- 1) Work in the standard PSL2(Z) presentation <S,T | S^2 = 1, (S*T)^3 = 1> ---
    # Here S corresponds to s2 = [[0,-1],[1,0]] and T corresponds to l = [[1,1],[0,1]].
    F.<S,T> = FreeGroup()
    PSL2 = F / [S**2, (S*T)**3]

    # We are given a helper that solves the word problem in SL2Z in terms of (l,r).
    # Convert each generator to a word in (S,T) using:
    #   l = T
    #   r = S*T^-1*S   (projectively; signs don't matter in PSL2)
    def matrix_to_PSL2_word(A):
        lr_word = sl2z_word_problem(A)   # list of pairs (0 or 1, exponent), 0=l, 1=r
        w = F.one()
        for which, exp in lr_word:
            if which == 0:      # l
                w *= T**exp
            elif which == 1:    # r = S*T^-1*S
                w *= (S * T**(-1) * S)**exp
            else:
                raise RuntimeError(f"Unexpected generator index {which} in sl2z_word_problem output.")
        return PSL2(w)

    H_words = [matrix_to_PSL2_word(A) for A in gensM]

    # --- 2) Coset enumeration in GAP: get the action of S and T on right cosets of H ---
    PSL2_gap = PSL2.gap()
    H_gap = libgap.Subgroup(PSL2_gap, [w.gap() for w in H_words])

    # GAP returns a "coset table" as lists of images; apply PermList to get permutations. :contentReference[oaicite:2]{index=2}
    ct = libgap.CosetTableFpGroup(PSL2_gap, H_gap)   # list-of-lists: gens then inverses

    # Our PSL2 presentation has exactly 2 generators: S and T.
    permS = libgap.PermList(ct[0]).sage()  # action of S on cosets
    permT = libgap.PermList(ct[1]).sage()  # action of T (= L) on cosets

    # --- 3) Build the arithmetic subgroup from permutations of S2 and L ---
    H = ArithmeticSubgroup_Permutation(S2=permS, L=permT, relabel=relabel, check=True)

    if sanity_check:
        # Check: each input generator (or its negative, which is same in PSL2) lands in the subgroup.
        # For even subgroups this is usually direct; for odd lifts, projectively you may need ±.
        for A in gensM:
            if (A not in H) and ((-A) not in H):
                raise RuntimeError(f"Sanity check failed: generator not recognized in constructed subgroup:\n{A}")

    # --- 4) Farey symbol + convenient invariants ---
    Fsym = H.farey_symbol()  # preferred public API in Sage 10.8 :contentReference[oaicite:3]{index=3}
    return H, Fsym

# -------------------------
# Example usage:
# -------------------------
# S2 = matrix(ZZ, [[0,-1],[1,0]])
# T  = matrix(ZZ, [[1,1],[0,1]])
# H, Fsym = subgroup_from_matrix_gens_farey([T**2, S2*T*S2])  # example gens
# print(H.index(), H.ncusps(), H.nu2(), H.nu3(), H.genus())
# print(Fsym)   # prints FareySymbol(...)

S2 = matrix(ZZ, [[0,-1],[1,0]])
T  = matrix(ZZ, [[1,1],[0,1]])
H, Fsym = subgroup_from_matrix_gens_farey([T**2, S2*T*S2])  # example gens
print(H.index(), H.ncusps(), H.nu2(), H.nu3(), H.genus())
print(Fsym)   # prints FareySymbol(...)
